# 피처 엔지니어링 실험 노트북

원본 피처와 문헌 기반 엔지니어링 피처를 같은 조건에서 비교합니다.


In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from time import perf_counter

import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, RobustScaler
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier


ROOT = Path("/home/ms990728/소정해")
DATA_PATH = ROOT / "data/raw/cell2celltrain.csv"
RESULT_PATH = ROOT / "results/baseline_results.md"
RANDOM_STATE = 42


@dataclass
class ResultRow:
    model: str
    train_seconds: float
    roc_auc: float
    accuracy: float
    precision: float
    recall: float
    f1: float


def load_data(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path, na_values=["NA", ""], keep_default_na=True)
    # HandsetPrice is stored as strings because of "Unknown" placeholders.
    df["HandsetPrice"] = pd.to_numeric(
        df["HandsetPrice"].replace("Unknown", pd.NA), errors="coerce"
    )
    df["Churn"] = df["Churn"].map({"No": 0, "Yes": 1}).astype("Int64")
    return df


def build_feature_sets(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.Series, list[str], list[str]]:
    feature_df = df.drop(columns=["CustomerID", "Churn"])
    target = df["Churn"].astype(int)

    numeric_cols = [
        col for col in feature_df.columns if pd.api.types.is_numeric_dtype(feature_df[col])
    ]
    categorical_cols = [col for col in feature_df.columns if col not in numeric_cols]

    return feature_df, target, numeric_cols, categorical_cols


def build_logistic_pipeline(numeric_cols: list[str], categorical_cols: list[str]) -> Pipeline:
    numeric_pipe = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", RobustScaler()),
        ]
    )
    categorical_pipe = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=True)),
        ]
    )
    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_pipe, numeric_cols),
            ("cat", categorical_pipe, categorical_cols),
        ]
    )

    model = LogisticRegression(
        max_iter=10000,
        solver="saga",
        class_weight="balanced",
        random_state=RANDOM_STATE,
    )

    return Pipeline(steps=[("preprocess", preprocessor), ("model", model)])


def build_tree_pipeline(
    numeric_cols: list[str],
    categorical_cols: list[str],
    estimator,
) -> Pipeline:
    numeric_pipe = Pipeline(steps=[("imputer", SimpleImputer(strategy="median"))])
    categorical_pipe = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            (
                "ordinal",
                OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1),
            ),
        ]
    )
    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_pipe, numeric_cols),
            ("cat", categorical_pipe, categorical_cols),
        ]
    )
    return Pipeline(steps=[("preprocess", preprocessor), ("model", estimator)])


def evaluate_model(name: str, pipeline: Pipeline, X_train, y_train, X_test, y_test) -> ResultRow:
    start = perf_counter()
    pipeline.fit(X_train, y_train)
    train_seconds = perf_counter() - start

    y_prob = pipeline.predict_proba(X_test)[:, 1]
    y_pred = (y_prob >= 0.5).astype(int)

    return ResultRow(
        model=name,
        train_seconds=train_seconds,
        roc_auc=roc_auc_score(y_test, y_prob),
        accuracy=accuracy_score(y_test, y_pred),
        precision=precision_score(y_test, y_pred, zero_division=0),
        recall=recall_score(y_test, y_pred, zero_division=0),
        f1=f1_score(y_test, y_pred, zero_division=0),
    )


def format_results(results: list[ResultRow]) -> str:
    header = (
        "| Model | Train sec | ROC-AUC | Accuracy | Precision | Recall | F1 |\n"
        "|---|---:|---:|---:|---:|---:|---:|\n"
    )
    rows = [
        f"| {r.model} | {r.train_seconds:.1f} | {r.roc_auc:.4f} | {r.accuracy:.4f} | "
        f"{r.precision:.4f} | {r.recall:.4f} | {r.f1:.4f} |"
        for r in results
    ]
    return header + "\n".join(rows) + "\n"


def main() -> None:
    df = load_data(DATA_PATH)
    feature_df, target, numeric_cols, categorical_cols = build_feature_sets(df)

    X_train, X_test, y_train, y_test = train_test_split(
        feature_df,
        target,
        test_size=0.2,
        stratify=target,
        random_state=RANDOM_STATE,
    )

    pos = int(y_train.sum())
    neg = int((y_train == 0).sum())
    scale_pos_weight = neg / max(pos, 1)

    logistic = build_logistic_pipeline(numeric_cols, categorical_cols)
    rf = build_tree_pipeline(
        numeric_cols,
        categorical_cols,
        RandomForestClassifier(
            n_estimators=300,
            min_samples_leaf=2,
            class_weight="balanced_subsample",
            random_state=RANDOM_STATE,
            n_jobs=-1,
        ),
    )
    xgb = build_tree_pipeline(
        numeric_cols,
        categorical_cols,
        XGBClassifier(
            n_estimators=300,
            max_depth=5,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_lambda=1.0,
            objective="binary:logistic",
            eval_metric="logloss",
            tree_method="hist",
            scale_pos_weight=scale_pos_weight,
            random_state=RANDOM_STATE,
            n_jobs=-1,
        ),
    )

    results = [
        evaluate_model("Logistic Regression", logistic, X_train, y_train, X_test, y_test),
        evaluate_model("Random Forest", rf, X_train, y_train, X_test, y_test),
        evaluate_model("XGBoost", xgb, X_train, y_train, X_test, y_test),
    ]

    summary = [
        "# Baseline Experiment Results",
        "",
        f"- Dataset: `{DATA_PATH.name}`",
        f"- Rows: `{len(df):,}`",
        f"- Features used: `{len(feature_df.columns)}`",
        f"- Numeric features: `{len(numeric_cols)}`",
        f"- Categorical features: `{len(categorical_cols)}`",
        f"- Train set size: `{len(X_train):,}`",
        f"- Test set size: `{len(X_test):,}`",
        f"- Positive class weight used for XGBoost: `{scale_pos_weight:.4f}`",
        "",
        format_results(results),
        "",
        "## Notes",
        "",
        "- `HandsetPrice` was converted to numeric with `Unknown` treated as missing.",
        "- Logistic Regression uses one-hot encoding for categorical variables.",
        "- Random Forest and XGBoost use ordinal encoding for categorical variables.",
        "- Metrics are computed on the held-out 20% test split.",
        "",
    ]

    text = "\n".join(summary)
    RESULT_PATH.write_text(text, encoding="utf-8")
    print(text)


In [ ]:
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier


RESULT_PATH = ROOT / "results/feature_engineering_results.md"


def infer_columns(feature_df: pd.DataFrame) -> tuple[list[str], list[str]]:
    numeric_cols = [
        col for col in feature_df.columns if pd.api.types.is_numeric_dtype(feature_df[col])
    ]
    categorical_cols = [col for col in feature_df.columns if col not in numeric_cols]
    return numeric_cols, categorical_cols


def engineer_features(feature_df: pd.DataFrame) -> pd.DataFrame:
    df = feature_df.copy()

    # Cost-shock and service-quality features used as our engineered feature set.
    df["PositiveRevenueShock"] = df["PercChangeRevenues"].clip(lower=0)
    df["RevenuePerTenure"] = df["MonthlyRevenue"] / (df["MonthsInService"] + 1)
    df["CallDropRate"] = df["DroppedCalls"] / (
        df["DroppedCalls"] + df["BlockedCalls"] + df["UnansweredCalls"] + 1
    )
    df["ServiceIssueIndex"] = df["DroppedCalls"] + df["BlockedCalls"] + df["CustomerCareCalls"]
    df["Overage_X_PosRevenue"] = df["OverageMinutes"] * df["PositiveRevenueShock"]
    df["Roaming_X_PosRevenue"] = df["RoamingCalls"] * df["PositiveRevenueShock"]
    df["Overage_X_Roaming"] = df["OverageMinutes"] * df["RoamingCalls"]
    return df


def run_experiment(
    feature_df: pd.DataFrame,
    target: pd.Series,
    feature_set_name: str,
    train_idx: np.ndarray,
    test_idx: np.ndarray,
) -> list[ResultRow]:
    X_train = feature_df.iloc[train_idx].copy()
    X_test = feature_df.iloc[test_idx].copy()
    y_train = target.iloc[train_idx]
    y_test = target.iloc[test_idx]

    numeric_cols, categorical_cols = infer_columns(feature_df)
    pos = int(y_train.sum())
    neg = int((y_train == 0).sum())
    scale_pos_weight = neg / max(pos, 1)

    logistic = build_logistic_pipeline(numeric_cols, categorical_cols)
    rf = build_tree_pipeline(
        numeric_cols,
        categorical_cols,
        RandomForestClassifier(
            n_estimators=300,
            min_samples_leaf=2,
            class_weight="balanced_subsample",
            random_state=RANDOM_STATE,
            n_jobs=-1,
        ),
    )
    xgb = build_tree_pipeline(
        numeric_cols,
        categorical_cols,
        XGBClassifier(
            n_estimators=300,
            max_depth=5,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_lambda=1.0,
            objective="binary:logistic",
            eval_metric="logloss",
            tree_method="hist",
            scale_pos_weight=scale_pos_weight,
            random_state=RANDOM_STATE,
            n_jobs=-1,
        ),
    )

    return [
        evaluate_model(
            f"{feature_set_name} / Logistic Regression",
            logistic,
            X_train,
            y_train,
            X_test,
            y_test,
        ),
        evaluate_model(
            f"{feature_set_name} / Random Forest",
            rf,
            X_train,
            y_train,
            X_test,
            y_test,
        ),
        evaluate_model(
            f"{feature_set_name} / XGBoost",
            xgb,
            X_train,
            y_train,
            X_test,
            y_test,
        ),
    ]


def format_table(results: list[ResultRow]) -> str:
    header = (
        "| Model | Train sec | ROC-AUC | Accuracy | Precision | Recall | F1 |\n"
        "|---|---:|---:|---:|---:|---:|---:|\n"
    )
    rows = [
        f"| {r.model} | {r.train_seconds:.1f} | {r.roc_auc:.4f} | {r.accuracy:.4f} | "
        f"{r.precision:.4f} | {r.recall:.4f} | {r.f1:.4f} |"
        for r in results
    ]
    return header + "\n".join(rows)


def main() -> None:
    df = load_data(DATA_PATH)
    base_features = df.drop(columns=["CustomerID", "Churn"])
    target = df["Churn"].astype(int)

    engineered_features = engineer_features(base_features)

    split_indices = np.arange(len(df))
    train_idx, test_idx = train_test_split(
        split_indices,
        test_size=0.2,
        stratify=target,
        random_state=RANDOM_STATE,
    )

    base_results = run_experiment(
        base_features,
        target,
        "Base",
        train_idx,
        test_idx,
    )
    engineered_results = run_experiment(
        engineered_features,
        target,
        "Engineered",
        train_idx,
        test_idx,
    )

    all_results = base_results + engineered_results

    by_model = {}
    for r in all_results:
        key = r.model.split(" / ")[1]
        by_model.setdefault(key, {})[r.model.split(" / ")[0]] = r

    delta_rows = []
    for model_name, rows in by_model.items():
        base = rows["Base"]
        eng = rows["Engineered"]
        delta_rows.append(
            {
                "Model": model_name,
                "ROC-AUC Δ": eng.roc_auc - base.roc_auc,
                "Accuracy Δ": eng.accuracy - base.accuracy,
                "F1 Δ": eng.f1 - base.f1,
            }
        )

    delta_df = pd.DataFrame(delta_rows).sort_values("Model")

    delta_header = "| Model | ROC-AUC Δ | Accuracy Δ | F1 Δ |\n|---|---:|---:|---:|"
    delta_rows_text = [
        f"| {row['Model']} | {row['ROC-AUC Δ']:.4f} | {row['Accuracy Δ']:.4f} | {row['F1 Δ']:.4f} |"
        for _, row in delta_df.iterrows()
    ]
    delta_table = "\n".join([delta_header, *delta_rows_text])

    summary = [
        "# Feature Engineering Experiment Results",
        "",
        f"- Dataset: `{DATA_PATH.name}`",
        f"- Rows: `{len(df):,}`",
        f"- Base features: `{len(base_features.columns)}`",
        f"- Engineered features added: `7`",
        f"- Train set size: `{len(train_idx):,}`",
        f"- Test set size: `{len(test_idx):,}`",
        "",
        "## Base Features",
        "",
        format_table(base_results),
        "",
        "## Engineered Features",
        "",
        format_table(engineered_results),
        "",
        "## Engineered Minus Base",
        "",
        delta_table,
        "",
        "## Engineered Feature Definitions",
        "",
        "- `PositiveRevenueShock = max(PercChangeRevenues, 0)`",
        "- `RevenuePerTenure = MonthlyRevenue / (MonthsInService + 1)`",
        "- `CallDropRate = DroppedCalls / (DroppedCalls + BlockedCalls + UnansweredCalls + 1)`",
        "- `ServiceIssueIndex = DroppedCalls + BlockedCalls + CustomerCareCalls`",
        "- `Overage_X_PosRevenue = OverageMinutes × PositiveRevenueShock`",
        "- `Roaming_X_PosRevenue = RoamingCalls × PositiveRevenueShock`",
        "- `Overage_X_Roaming = OverageMinutes × RoamingCalls`",
        "",
    ]

    text = "\n".join(summary)
    RESULT_PATH.write_text(text, encoding="utf-8")
    print(text)


if __name__ == "__main__":
    main()


# Feature Engineering Experiment Results

- Dataset: `cell2celltrain.csv`
- Rows: `51,047`
- Base features: `56`
- Engineered features added: `7`
- Train set size: `40,837`
- Test set size: `10,210`

## Base Features

| Model | Train sec | ROC-AUC | Accuracy | Precision | Recall | F1 |
|---|---:|---:|---:|---:|---:|---:|
| Base / Logistic Regression | 50.2 | 0.6084 | 0.5848 | 0.3625 | 0.5809 | 0.4464 |
| Base / Random Forest | 1.9 | 0.6676 | 0.7200 | 0.5737 | 0.1098 | 0.1843 |
| Base / XGBoost | 0.7 | 0.6820 | 0.6309 | 0.4087 | 0.6292 | 0.4955 |

## Engineered Features

| Model | Train sec | ROC-AUC | Accuracy | Precision | Recall | F1 |
|---|---:|---:|---:|---:|---:|---:|
| Engineered / Logistic Regression | 114.4 | 0.5215 | 0.4507 | 0.3017 | 0.6893 | 0.4197 |
| Engineered / Random Forest | 1.9 | 0.6668 | 0.7177 | 0.5579 | 0.0982 | 0.1671 |
| Engineered / XGBoost | 0.6 | 0.6794 | 0.6305 | 0.4073 | 0.6203 | 0.4917 |

## Engineered Minus Base

| Model | ROC-AUC Δ | Accuracy Δ | F1 Δ |
|---|---:|---:|---:|
| Logistic Regression | -0.0868 | -0.1341 | -0.0267 |
| Random Forest | -0.0008 | -0.0023 | -0.0173 |
| XGBoost | -0.0027 | -0.0004 | -0.0038 |

## Engineered Feature Definitions

- `PositiveRevenueShock = max(PercChangeRevenues, 0)`
- `RevenuePerTenure = MonthlyRevenue / (MonthsInService + 1)`
- `CallDropRate = DroppedCalls / (DroppedCalls + BlockedCalls + UnansweredCalls + 1)`
- `ServiceIssueIndex = DroppedCalls + BlockedCalls + CustomerCareCalls`
- `Overage_X_PosRevenue = OverageMinutes × PositiveRevenueShock`
- `Roaming_X_PosRevenue = RoamingCalls × PositiveRevenueShock`
- `Overage_X_Roaming = OverageMinutes × RoamingCalls`
